In [117]:
import os
import copy
import numpy as np
from langchain_community.document_loaders import PyPDFLoader, Docx2txtLoader
from langchain_text_splitters.character import CharacterTextSplitter
from langchain_text_splitters.markdown import MarkdownHeaderTextSplitter
from langchain_core.documents import Document
from langchain_ollama.embeddings import OllamaEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_core.runnables import RunnableParallel, RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import PromptTemplate
from langchain_ollama import ChatOllama

# Indexing: Document Loading with PyPDF Loader

In [2]:
loader_pdf = PyPDFLoader("Introduction_to_Data_and_Data_Science.pdf")

In [3]:
pages_pdf = loader_pdf.load()

In [4]:
pages_pdf

[Document(metadata={'producer': 'Microsoft® Word for Microsoft 365', 'creator': 'Microsoft® Word for Microsoft 365', 'creationdate': '2023-11-09T10:16:34+02:00', 'author': 'Hristina  Hristova', 'moddate': '2023-11-09T10:16:34+02:00', 'source': 'Introduction_to_Data_and_Data_Science.pdf', 'total_pages': 6, 'page': 0, 'page_label': '1'}, page_content='Analysis vs Analytics \nAlright! So… \nLet’s discuss the not-so-obvious differences \nbetween the terms analysis and analytics. \nDue to the similarity of the words, some people \nbelieve they share the same meaning, and thus \nuse them interchangeably. Technically, this \nisn’t correct. There is, in fact, a distinct \ndifference between the two. And the reason \nfor one often being used instead of the other \nis the lack of a transparent understanding \nof both. \nSo, let’s clear this up, shall we? \nFirst, we will start with analysis. \nConsider the following… \nYou have a huge dataset containing data of \nvarious types. Instead of tackli

In [5]:
pages_pdf_cut = copy.deepcopy(pages_pdf)

In [6]:
" ".join(pages_pdf_cut[0].page_content.split() )

'Analysis vs Analytics Alright! So… Let’s discuss the not-so-obvious differences between the terms analysis and analytics. Due to the similarity of the words, some people believe they share the same meaning, and thus use them interchangeably. Technically, this isn’t correct. There is, in fact, a distinct difference between the two. And the reason for one often being used instead of the other is the lack of a transparent understanding of both. So, let’s clear this up, shall we? First, we will start with analysis. Consider the following… You have a huge dataset containing data of various types. Instead of tackling the entire dataset and running the risk of becoming overwhelmed, you separate it into easier to digest chunks and study them individually and examine how they relate to other parts. And that’s analysis in a nutshell. One important thing to remember, however, is that you perform analyses on things that have already happened in the past. Such as using an analysis to explain how a

In [7]:
for i in pages_pdf_cut:
    i.page_content = " ".join(i.page_content.split() )

In [8]:
pages_pdf_cut

[Document(metadata={'producer': 'Microsoft® Word for Microsoft 365', 'creator': 'Microsoft® Word for Microsoft 365', 'creationdate': '2023-11-09T10:16:34+02:00', 'author': 'Hristina  Hristova', 'moddate': '2023-11-09T10:16:34+02:00', 'source': 'Introduction_to_Data_and_Data_Science.pdf', 'total_pages': 6, 'page': 0, 'page_label': '1'}, page_content='Analysis vs Analytics Alright! So… Let’s discuss the not-so-obvious differences between the terms analysis and analytics. Due to the similarity of the words, some people believe they share the same meaning, and thus use them interchangeably. Technically, this isn’t correct. There is, in fact, a distinct difference between the two. And the reason for one often being used instead of the other is the lack of a transparent understanding of both. So, let’s clear this up, shall we? First, we will start with analysis. Consider the following… You have a huge dataset containing data of various types. Instead of tackling the entire dataset and runnin

# Indexing: Document Loading with DOCX2TXT Loader

In [9]:
loader_docx = Docx2txtLoader("Introduction_to_Data_and_Data_Science.docx")

In [10]:
pages_docx = loader_docx.load()

In [11]:
pages_docx

[Document(metadata={'source': 'Introduction_to_Data_and_Data_Science.docx'}, page_content="Analysis vs Analytics\n\nAlright! So…\nLet’s discuss the not-so-obvious differences\nbetween the terms analysis and analytics.\nDue to the similarity of the words, some people\nbelieve they share the same meaning, and thus\nuse them interchangeably. Technically, this\nisn’t correct. There is, in fact, a distinct\ndifference between the two. And the reason\nfor one often being used instead of the other\nis the lack of a transparent understanding\nof both.\nSo, let’s clear this up, shall we?\nFirst, we will start with analysis.\nConsider the following…\nYou have a huge dataset containing data of\nvarious types. Instead of tackling the entire\ndataset and running the risk of becoming overwhelmed,\nyou separate it into easier to digest chunks\nand study them individually and examine how\nthey relate to other parts. And that’s analysis\nin a nutshell.\nOne important thing to remember, however,\nis tha

# Indexing: Document Splitting with Character Text Splittier

In [12]:
loader = Docx2txtLoader("Introduction_to_Data_and_Data_Science.docx")

In [13]:
pages = loader.load()

In [14]:
for i in range(len(pages)):
    pages[i].page_content = " ".join(pages[i].page_content.split()) 

In [15]:
pages[0].page_content

"Analysis vs Analytics Alright! So… Let’s discuss the not-so-obvious differences between the terms analysis and analytics. Due to the similarity of the words, some people believe they share the same meaning, and thus use them interchangeably. Technically, this isn’t correct. There is, in fact, a distinct difference between the two. And the reason for one often being used instead of the other is the lack of a transparent understanding of both. So, let’s clear this up, shall we? First, we will start with analysis. Consider the following… You have a huge dataset containing data of various types. Instead of tackling the entire dataset and running the risk of becoming overwhelmed, you separate it into easier to digest chunks and study them individually and examine how they relate to other parts. And that’s analysis in a nutshell. One important thing to remember, however, is that you perform analyses on things that have already happened in the past. Such as using an analysis to explain how a

In [16]:
len(pages[0].page_content)

8259

In [17]:
char_splitter = CharacterTextSplitter(separator= ".", chunk_size = 500, chunk_overlap =50)

In [18]:
pages_chat_split = char_splitter.split_documents(pages)

In [19]:
pages_chat_split

[Document(metadata={'source': 'Introduction_to_Data_and_Data_Science.docx'}, page_content='Analysis vs Analytics Alright! So… Let’s discuss the not-so-obvious differences between the terms analysis and analytics. Due to the similarity of the words, some people believe they share the same meaning, and thus use them interchangeably. Technically, this isn’t correct. There is, in fact, a distinct difference between the two. And the reason for one often being used instead of the other is the lack of a transparent understanding of both'),
 Document(metadata={'source': 'Introduction_to_Data_and_Data_Science.docx'}, page_content='So, let’s clear this up, shall we? First, we will start with analysis. Consider the following… You have a huge dataset containing data of various types. Instead of tackling the entire dataset and running the risk of becoming overwhelmed, you separate it into easier to digest chunks and study them individually and examine how they relate to other parts. And that’s anal

In [20]:
len(pages_chat_split)

21

In [21]:
8259 / 450


18.35333333333333

In [22]:
0.353 * 450

158.85

In [23]:
len(pages_chat_split[0].page_content)

444

# Document Splitting with Markdown Header Text Splitter

In [24]:
loader_docx = Docx2txtLoader("Introduction_to_Data_and_Data_Science_2.docx")

In [25]:
pages_docx = loader_docx.load()

In [26]:
pages_docx

[Document(metadata={'source': 'Introduction_to_Data_and_Data_Science_2.docx'}, page_content="# Introduction to Data and Data Science\n\n## Analysis vs Analytics\n\nAlright! So…\nLet’s discuss the not-so-obvious differences\nbetween the terms analysis and analytics.\nDue to the similarity of the words, some people\nbelieve they share the same meaning, and thus\nuse them interchangeably. Technically, this\nisn’t correct. There is, in fact, a distinct\ndifference between the two. And the reason\nfor one often being used instead of the other\nis the lack of a transparent understanding\nof both.\nSo, let’s clear this up, shall we?\nFirst, we will start with analysis.\nConsider the following…\nYou have a huge dataset containing data of\nvarious types. Instead of tackling the entire\ndataset and running the risk of becoming overwhelmed,\nyou separate it into easier to digest chunks\nand study them individually and examine how\nthey relate to other parts. And that’s analysis\nin a nutshell.\nO

In [27]:
md_splitter = MarkdownHeaderTextSplitter(headers_to_split_on=[("#", "Course Title"), ("##", "Lecture Title")])

In [28]:
pages_md_splitter = md_splitter.split_text(pages_docx[0].page_content)

In [29]:
pages_md_splitter

[Document(metadata={'Course Title': 'Introduction to Data and Data Science', 'Lecture Title': 'Analysis vs Analytics'}, page_content="Alright! So…\nLet’s discuss the not-so-obvious differences\nbetween the terms analysis and analytics.\nDue to the similarity of the words, some people\nbelieve they share the same meaning, and thus\nuse them interchangeably. Technically, this\nisn’t correct. There is, in fact, a distinct\ndifference between the two. And the reason\nfor one often being used instead of the other\nis the lack of a transparent understanding\nof both.\nSo, let’s clear this up, shall we?\nFirst, we will start with analysis.\nConsider the following…\nYou have a huge dataset containing data of\nvarious types. Instead of tackling the entire\ndataset and running the risk of becoming overwhelmed,\nyou separate it into easier to digest chunks\nand study them individually and examine how\nthey relate to other parts. And that’s analysis\nin a nutshell.\nOne important thing to remember

# Indexing: Text Embedding with OllamaAI


In [30]:
loader_docx = Docx2txtLoader("Introduction_to_Data_and_Data_Science_2.docx")

In [31]:
pages_docx = loader_docx.load()

In [32]:
md_splitter = MarkdownHeaderTextSplitter(headers_to_split_on=[("#", "Course Title"), ("##", "Lecture Title")])

In [33]:
pages_md_splitter = md_splitter.split_text(pages_docx[0].page_content)

In [34]:
for i in range(len(pages_md_splitter)):
    pages_md_splitter[i].page_content = " ".join(pages_md_splitter[i].page_content.split())

In [35]:
md_splitter_char_splitter = CharacterTextSplitter(separator= ".", chunk_size = 500, chunk_overlap =50)

In [36]:
md_splitter_char_splitter

In [37]:
pages_md_splitter_char_splitter = md_splitter_char_splitter.split_documents(pages_md_splitter)

In [38]:
pages_md_splitter_char_splitter

[Document(metadata={'Course Title': 'Introduction to Data and Data Science', 'Lecture Title': 'Analysis vs Analytics'}, page_content='Alright! So… Let’s discuss the not-so-obvious differences between the terms analysis and analytics. Due to the similarity of the words, some people believe they share the same meaning, and thus use them interchangeably. Technically, this isn’t correct. There is, in fact, a distinct difference between the two. And the reason for one often being used instead of the other is the lack of a transparent understanding of both. So, let’s clear this up, shall we? First, we will start with analysis'),
 Document(metadata={'Course Title': 'Introduction to Data and Data Science', 'Lecture Title': 'Analysis vs Analytics'}, page_content='Consider the following… You have a huge dataset containing data of various types. Instead of tackling the entire dataset and running the risk of becoming overwhelmed, you separate it into easier to digest chunks and study them individu

In [39]:
embedding = OllamaEmbeddings(model= "nomic-embed-text")

In [40]:
vector1 = embedding.embed_query(pages_md_splitter_char_splitter[3].page_content)
vector2 = embedding.embed_query(pages_md_splitter_char_splitter[5].page_content)
vector3 = embedding.embed_query(pages_md_splitter_char_splitter[18].page_content)

In [41]:
len(vector1),len(vector2),len(vector3)

(768, 768, 768)

In [42]:
np.dot(vector1, vector2), np.dot(vector1, vector3),np.dot(vector2, vector3)

(0.7667608188729719, 0.6024198938523961, 0.5484319943387258)

In [43]:
np.linalg.norm(vector1),np.linalg.norm(vector2),np.linalg.norm(vector3)

(1.0000003701636613, 1.0000002608902259, 0.9999998027646158)

In [44]:
len(pages_md_splitter_char_splitter)

20

# Indexing Chroma Vector Store

In [45]:
loader_docx = Docx2txtLoader("Introduction_to_Data_and_Data_Science_2.docx")

In [46]:
pages_docx = loader_docx.load()

In [47]:
md_splitter = MarkdownHeaderTextSplitter(headers_to_split_on=[("#", "Course Title"), ("##", "Lecture Title")])

In [48]:
pages_md_split = md_splitter.split_text(pages_docx[0].page_content)

In [49]:
for i in range(len(pages_md_split)):
    pages_md_split[i].page_content = " ".join(pages_md_split[i].page_content.split())

In [50]:
char_splitter = CharacterTextSplitter(separator= ".", chunk_size = 500, chunk_overlap =50)

In [54]:
pages_char_splitter = char_splitter.split_documents(pages_md_splitter)

In [51]:
embedding = OllamaEmbeddings(model= "nomic-embed-text")

In [55]:
vectorstore = Chroma.from_documents(documents=pages_char_splitter, embedding=embedding, persist_directory= "./intro-to-ds-lectures")

In [56]:
vectorstore_from_directory = Chroma(persist_directory= "./intro-to-ds-lectures", embedding_function=embedding)

/var/folders/l0/86zkjfks5gz5k5rw35hv8n180000gn/T/ipykernel_87278/4145202912.py:1: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-chroma package and should be used instead. To use it run `pip install -U :class:`~langchain-chroma` and import as `from :class:`~langchain_chroma import Chroma``.
  vectorstore_from_directory = Chroma(persist_directory= "./intro-to-ds-lectures", embedding_function=embedding)


# Indexing: Inspecting and Managing Documents in a Vectorstore

In [57]:
vectorstore = Chroma.from_documents(documents=pages_char_splitter, embedding=embedding, persist_directory= "./intro-to-ds-lectures")

In [58]:
vectorstore_from_directory = Chroma(persist_directory= "./intro-to-ds-lectures", embedding_function=embedding)

In [59]:
vectorstore_from_directory.get()

{'ids': ['49fede35-9296-48c2-8906-35ca31308490',
  '32ef3f08-78af-4a81-82e4-d0f6f2a720ba',
  '7fe56362-5f97-4cfe-99c3-32055706156c',
  '2284c83b-2440-49eb-bd6e-e8a7efd188b5',
  '9bb060d4-dfb1-4da6-9609-0e91340e1d11',
  '18cddda1-d4eb-4c45-a577-73e94f7a8c9b',
  'ee267d6d-daed-47b0-a1f4-4b4a1bed6975',
  'b378d644-fb2a-4204-a5cb-f781ac667b78',
  'df76707b-6488-4a94-9d15-639466253343',
  'f15e9d60-b62a-42ba-ad45-db062b2f9e41',
  '227a9ac8-6979-4576-bcbb-6fb53f369ae2',
  '7547fc62-9c28-4035-9d81-288df369bac7',
  '600f3b9c-2415-43de-b964-e5906d06cf03',
  '9bed8ee9-a5d9-4fdf-bc17-2675f7f0255a',
  '8eef1929-2027-4063-9159-9903fe44f0cf',
  '8e09b992-fbad-438e-bcea-722ad4f9c859',
  '1bb92755-e2f0-4131-9085-70a702eac3ac',
  'a26deb18-7668-435c-a340-61c42bdf7937',
  'a90d1494-8032-4a9e-9523-880705f7ebde',
  'a391ff08-5f55-4215-a5bb-292819c12635',
  'b7daccc1-bdf1-4def-bad0-7e145fb4ee9b',
  'aa80f379-cc7e-454d-ac73-5e7ab6709150',
  '4b8a0ca4-e813-42a3-b7c2-43751bc61be5',
  '5cbeda88-677e-4840-80e0-

In [66]:
added_document = Document(page_content='Alright! So… Let’s discuss the not-so-obvious differences between the terms analysis and analytics. Due to the similarity of the words, some people believe they share the same meaning, and thus use them interchangeably. Technically, this isn’t correct. There is, in fact, a distinct difference between the two. And the reason for one often being used instead of the other is the lack of a transparent understanding of both. So, let’s clear this up, shall we? First, we will start with analysis', 
                          metadata={'Course Title': 'Introduction to Data and Data Science', 
                                    'Lecture Title': 'Analysis vs Analytics'})

In [69]:
vectorstore_from_directory.add_documents([added_document])

['3312f589-0cec-44db-9422-ad326d34860f']

In [70]:
vectorstore_from_directory.get("3312f589-0cec-44db-9422-ad326d34860f")

{'ids': ['3312f589-0cec-44db-9422-ad326d34860f'],
 'embeddings': None,
 'documents': ['Alright! So… Let’s discuss the not-so-obvious differences between the terms analysis and analytics. Due to the similarity of the words, some people believe they share the same meaning, and thus use them interchangeably. Technically, this isn’t correct. There is, in fact, a distinct difference between the two. And the reason for one often being used instead of the other is the lack of a transparent understanding of both. So, let’s clear this up, shall we? First, we will start with analysis'],
 'uris': None,
 'included': ['metadatas', 'documents'],
 'data': None,
 'metadatas': [{'Lecture Title': 'Analysis vs Analytics',
   'Course Title': 'Introduction to Data and Data Science'}]}

In [71]:
updated_document = Document(page_content='Great! We hope we gave you a good idea about the level of applicability of the most frequently used programming and software tools in the field of data science. Thank you for watching!', 
                            metadata={'Course Title': 'Introduction to Data and Data Science', 
                                     'Lecture Title': 'Programming Languages & Software Employed in Data Science - All the Tools You Need'})

In [72]:
vectorstore_from_directory.update_document(document_id= "3312f589-0cec-44db-9422-ad326d34860f", document= updated_document)

In [73]:
vectorstore_from_directory.get("3312f589-0cec-44db-9422-ad326d34860f")

{'ids': ['3312f589-0cec-44db-9422-ad326d34860f'],
 'embeddings': None,
 'documents': ['Great! We hope we gave you a good idea about the level of applicability of the most frequently used programming and software tools in the field of data science. Thank you for watching!'],
 'uris': None,
 'included': ['metadatas', 'documents'],
 'data': None,
 'metadatas': [{'Course Title': 'Introduction to Data and Data Science',
   'Lecture Title': 'Programming Languages & Software Employed in Data Science - All the Tools You Need'}]}

In [74]:
vectorstore_from_directory.delete("3312f589-0cec-44db-9422-ad326d34860f")

In [75]:
vectorstore_from_directory.get("3312f589-0cec-44db-9422-ad326d34860f")

{'ids': [],
 'embeddings': None,
 'documents': [],
 'uris': None,
 'included': ['metadatas', 'documents'],
 'data': None,
 'metadatas': []}

# Retrieval : Similarity Search

In [77]:
embedding = OllamaEmbeddings(model= "nomic-embed-text")

In [78]:
vectorstore = Chroma.from_documents(documents=pages_char_splitter, embedding=embedding, persist_directory= "./intro-to-ds-lectures")

In [79]:
added_document = Document(page_content='Alright! So… Let’s discuss the not-so-obvious differences between the terms analysis and analytics. Due to the similarity of the words, some people believe they share the same meaning, and thus use them interchangeably. Technically, this isn’t correct. There is, in fact, a distinct difference between the two. And the reason for one often being used instead of the other is the lack of a transparent understanding of both. So, let’s clear this up, shall we? First, we will start with analysis', 
                          metadata={'Course Title': 'Introduction to Data and Data Science', 
                                    'Lecture Title': 'Analysis vs Analytics'})

In [82]:
vectorstore.add_documents([added_document])

['b7146f2f-9ac3-46f6-8933-9b587b60530f']

In [83]:
question = "What programming language data scienetists use?"

In [84]:
retrieved_docs = vectorstore.similarity_search(query= question, k = 5)

In [85]:
retrieved_docs

[Document(metadata={'Course Title': 'Introduction to Data and Data Science', 'Lecture Title': 'Programming Languages & Software Employed in Data Science - All the Tools You Need'}, page_content='Thus, we need a lot of computational power, and we can expect people to use the languages similar to those in the big data column. Apart from R, Python, and MATLAB, other, faster languages are used like Java, JavaScript, C, C++, and Scala. Cool. What we said may be wonderful, but that’s not all! By using one or more programming languages, people create application software or, as they are sometimes called, software solutions, that are adjusted for specific business needs'),
 Document(metadata={'Course Title': 'Introduction to Data and Data Science', 'Lecture Title': 'Programming Languages & Software Employed in Data Science - All the Tools You Need'}, page_content='Thus, we need a lot of computational power, and we can expect people to use the languages similar to those in the big data column. 

In [90]:
for i in retrieved_docs:
    print(f"Page content:{i.page_content} \n \nLecture Title:{i.metadata['Lecture Title']} \n")

Page content:Thus, we need a lot of computational power, and we can expect people to use the languages similar to those in the big data column. Apart from R, Python, and MATLAB, other, faster languages are used like Java, JavaScript, C, C++, and Scala. Cool. What we said may be wonderful, but that’s not all! By using one or more programming languages, people create application software or, as they are sometimes called, software solutions, that are adjusted for specific business needs 
 
Lecture Title:Programming Languages & Software Employed in Data Science - All the Tools You Need 

Page content:Thus, we need a lot of computational power, and we can expect people to use the languages similar to those in the big data column. Apart from R, Python, and MATLAB, other, faster languages are used like Java, JavaScript, C, C++, and Scala. Cool. What we said may be wonderful, but that’s not all! By using one or more programming languages, people create application software or, as they are some

In [96]:
question = "What software do data scienetists use?"

In [99]:
retrieved_docs = vectorstore.max_marginal_relevance_search(query= question, k = 5, lambda_mult= 0.1)

In [100]:
for i in retrieved_docs:
    print(f"Page content:{i.page_content} \n \nLecture Title:{i.metadata['Lecture Title']} \n")

Page content:Great! We hope we gave you a good idea about the level of applicability of the most frequently used programming and software tools in the field of data science. Thank you for watching! 
 
Lecture Title:Programming Languages & Software Employed in Data Science - All the Tools You Need 

Page content:Alright! So… How are the techniques used in data, business intelligence, or predictive analytics applied in real life? Certainly, with the help of computers. You can basically split the relevant tools into two categories—programming languages and software. Knowing a programming language enables you to devise programs that can execute specific operations. Moreover, you can reuse these programs whenever you need to execute the same action 
 
Lecture Title:Programming Languages & Software Employed in Data Science - All the Tools You Need 

Page content:As you can see from the infographic, R, and Python are the two most popular tools across all columns. Their biggest advantage is th

# Retrieval: Vectorstore-Backed Retriever

In [102]:
embedding = OllamaEmbeddings()

ValidationError: 1 validation error for OllamaEmbeddings
model
  Field required [type=missing, input_value={}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.11/v/missing

In [101]:
vectorstore = Chroma.from_documents(documents=pages_char_splitter, embedding=embedding, persist_directory= "./intro-to-ds-lectures")

In [105]:
len(vectorstore.get()['documents'])

81

In [106]:
retriver = vectorstore.as_retriever(search_type = 'mmr', search_kwargs = {'k':3, 'lambda_mult': 0.7})

In [107]:
retriver

VectorStoreRetriever(tags=['Chroma', 'OllamaEmbeddings'], vectorstore=<langchain_community.vectorstores.chroma.Chroma object at 0x17c28e690>, search_type='mmr', search_kwargs={'k': 3, 'lambda_mult': 0.7})

In [108]:
question = "What software do data scienetists use?"

In [109]:
retrieved_docs = retriver.invoke(question)

In [110]:
retrieved_docs

[Document(metadata={'Lecture Title': 'Programming Languages & Software Employed in Data Science - All the Tools You Need', 'Course Title': 'Introduction to Data and Data Science'}, page_content='Great! We hope we gave you a good idea about the level of applicability of the most frequently used programming and software tools in the field of data science. Thank you for watching!'),
 Document(metadata={'Course Title': 'Introduction to Data and Data Science', 'Lecture Title': 'Programming Languages & Software Employed in Data Science - All the Tools You Need'}, page_content='Alright! So… How are the techniques used in data, business intelligence, or predictive analytics applied in real life? Certainly, with the help of computers. You can basically split the relevant tools into two categories—programming languages and software. Knowing a programming language enables you to devise programs that can execute specific operations. Moreover, you can reuse these programs whenever you need to execu

In [111]:
for i in retrieved_docs:
    print(f"Page content:{i.page_content} \n \nLecture Title:{i.metadata['Lecture Title']} \n")

Page content:Great! We hope we gave you a good idea about the level of applicability of the most frequently used programming and software tools in the field of data science. Thank you for watching! 
 
Lecture Title:Programming Languages & Software Employed in Data Science - All the Tools You Need 

Page content:Alright! So… How are the techniques used in data, business intelligence, or predictive analytics applied in real life? Certainly, with the help of computers. You can basically split the relevant tools into two categories—programming languages and software. Knowing a programming language enables you to devise programs that can execute specific operations. Moreover, you can reuse these programs whenever you need to execute the same action 
 
Lecture Title:Programming Languages & Software Employed in Data Science - All the Tools You Need 

Page content:It’s actually a software framework which was designed to address the complexity of big data and its computational intensity. Most n

# Generation: Stuffing Documents

In [113]:
vectorstore = Chroma.from_documents(documents=pages_char_splitter, embedding=embedding, persist_directory= "./intro-to-ds-lectures")

In [114]:
retriver = vectorstore.as_retriever(search_type = 'mmr', search_kwargs = {'k':3, 'lambda_mult': 0.7})

In [115]:
TEMPLATE = '''
Answer the following question:
{question}

To answer the question, use only the following context:
{context}

At the end of the response, specify the name of the lecture this context is taken from in the format:
Resources: *Lecture Title*
where *Lecture Title* should be substituted with the title of all resource lectures.
'''

prompt_template = PromptTemplate.from_template(TEMPLATE)

In [118]:
chat = ChatOllama(
    model = "llama3.2:latest",
    validate_model_on_init = True,
    temperature = 0.8,
    num_predict = 256,
    seed= 365
)

In [119]:
question = "What software do data scienetists use?"

In [127]:
chain ={'context': retriver, 'question': RunnablePassthrough() } |  prompt_template | chat | StrOutputParser()

In [128]:
chain.invoke(question)

'Based on the provided context, data scientists use a variety of programming languages and software tools. Some examples mentioned include:\n\n1. Programming languages:\n   - No specific programming languages are listed in the given text.\n\n2. Software frameworks and tools:\n   - Hadoop: A software framework designed for handling big data.\n   - Power BI, SaS, Qlik, and Tableau: Top-notch examples of business intelligence visualizations and software.\n\nNo other specific software is mentioned in the provided context.\n\nResources: Programming Languages & Software Employed in Data Science - All the Tools You Need'